In [9]:
!pip install opencv-python dlib numpy
#!wget https://github.com/davisking/dlib-models/blob/master/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 shape_predictor_68_face_landmarks.dat.bz2

In [11]:
import cv2
import dlib
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, HTML
from google.colab import output
import base64
import os

# Crear una carpeta en Colab para almacenar las capturas
CARPETA_GUARDADO = "capturas_rostros"
os.makedirs(CARPETA_GUARDADO, exist_ok=True)

# Inicializar dlib
hog_face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Contador para los nombres de los archivos guardados
contador_imagenes = 0

def detectar_emocion(landmarks):
    """Calcula distancias geométricas simples para estimar la emoción."""
    comisaria_izquierda = landmarks.part(48).x
    comisaria_derecha = landmarks.part(54).x
    labio_superior = landmarks.part(51).y
    labio_inferior = landmarks.part(57).y
    ceja_izq_interna = landmarks.part(21).x
    ceja_der_interna = landmarks.part(22).x

    ancho_boca = comisaria_derecha - comisaria_izquierda
    altura_boca = labio_inferior - labio_superior
    distancia_cejas = ceja_der_interna - ceja_izq_interna

    if ancho_boca > 50 and altura_boca > 12:
        return "Feliz"
    elif distancia_cejas < 14:
        return "Enojado"
    elif altura_boca < 6:
        return "Triste"
    return "Neutral"

# Función puente de procesamiento que ahora guarda la imagen
def procesar_cuadro(img_base64):
    global contador_imagenes
    try:
        encabezado, datos = img_base64.split(',')
        img_bytes = base64.b64decode(datos)
        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if frame is None: return "Rostro no detectado"

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = hog_face_detector(gray)

        for rostro in rostros:
            x, y, w, h = rostro.left(), rostro.top(), rostro.width(), rostro.height()

            # Dibujar marcas en la imagen que se muestra en pantalla
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            landmarks = landmark_predictor(gray, rostro)

            for n in range(0, 68):
                cv2.circle(frame, (landmarks.part(n).x, landmarks.part(n).y), 2, (255, 0, 0), -1)

            emocion = detectar_emocion(landmarks)
            cv2.putText(frame, f"Emocion: {emocion}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            # ==========================================================
            # SECCIÓN DE GUARDADO DE LA IMAGEN
            # ==========================================================
            # Guardamos un fotograma cada 15 llamadas para no saturar el disco
            if contador_imagenes % 15 == 0:
                ruta_archivo = os.path.join(CARPETA_GUARDADO, f"rostro_{emocion}_{contador_imagenes}.jpg")
                # Guardamos el cuadro con el rectángulo HOG y la emoción dibujada
                cv2.imwrite(ruta_archivo, frame)
                print(f"\n[INFO] Imagen guardada en Colab: {ruta_archivo}")

            contador_imagenes += 1

        # Enviar la imagen procesada de vuelta a la interfaz Web
        _, buffer = cv2.imencode('.jpg', frame)
        img_as_text = base64.b64encode(buffer).decode('utf-8')
        return f"data:image/jpeg;base64,{img_as_text}"
    except Exception as e:
        return str(e)

output.register_callback('notebook.procesar_cuadro', procesar_cuadro)

# Interfaz de video usando HTML5
html_camara = HTML('''
    <div style="font-family: Arial, sans-serif; text-align: center; background: #222; padding: 15px; border-radius: 8px; color: white; max-width: 500px; margin: auto;">
        <h3>Detector HOG + Guardado de Imágenes</h3>
        <p style="font-size: 12px; color: #aaa;">Las fotos se guardarán automáticamente en la carpeta "capturas_rostros"</p>
        <video id="video" width="400" height="300" autoplay playsinline style="border: 2px solid #555; border-radius: 4px; display: none;"></video>
        <img id="output_image" width="400" height="300" style="border: 2px solid #00ff00; border-radius: 4px; background: #000;" />
        <br/><br/>
        <button id="btn-start" style="padding: 10px 20px; font-weight: bold; background: #28a745; color: white; border: none; border-radius: 4px; cursor: pointer;">Iniciar Cámara</button>
    </div>

    <script>
        const video = document.getElementById('video');
        const imgOutput = document.getElementById('output_image');
        const btnStart = document.getElementById('btn-start');
        let streamActive = false;
        let intervalId = null;

        async function initCamera() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 400, height: 300 } });
                video.srcObject = stream;
                streamActive = true;
                btnStart.textContent = "Detener Cámara";
                btnStart.style.background = "#dc3545";

                const canvas = document.createElement('canvas');
                canvas.width = 400;
                canvas.height = 300;
                const ctx = canvas.getContext('2d');

                intervalId = setInterval(async () => {
                    if(!streamActive) return;
                    ctx.drawImage(video, 0, 0, 400, 300);
                    const dataUrl = canvas.toDataURL('image/jpeg', 0.7);

                    const respuesta = await google.colab.kernel.invokeFunction('notebook.procesar_cuadro', [dataUrl], {});
                    if(respuesta && respuesta.data) {
                        imgOutput.src = respuesta.data['text/plain'].replace(/'/g, "");
                    }
                }, 150);

            } catch (err) {
                alert("Error al acceder a la cámara: " + err.message);
            }
        }

        btnStart.onclick = () => {
            if(!streamActive) {
                initCamera();
            } else {
                streamActive = false;
                clearInterval(intervalId);
                const tracks = video.srcObject.getTracks();
                tracks.forEach(track => track.stop());
                btnStart.textContent = "Iniciar Cámara";
                btnStart.style.background = "#28a745";
                imgOutput.src = "";
            }
        };
    </script>
''')

display(html_camara)



[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_0.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Feliz_15.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_30.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_45.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_60.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_75.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_90.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Feliz_105.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_120.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_135.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_150.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Feliz_165.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/rostro_Neutral_180.jpg

[INFO] Imagen guardada en Colab: capturas_rostros/r